In [1]:
import csv
import json
import os
import requests
from bs4 import BeautifulSoup

In [4]:
def scrape_data(url, max_items=10):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Lỗi truy cập trang web: {e}")
        return []

    soup = BeautifulSoup(response.content, "html.parser")
    articles = soup.find_all("article", class_="product_pod")

    data = []
    for item in articles[:max_items]:
        title = item.h3.a["title"].strip()
        price = item.find("p", class_="price_color").text.strip()
        availability = item.find(
            "p", class_="instock availability"
        ).text.strip()

        data.append(
            {"Title": title, "Price": price, "Availability": availability}
        )

    return data

In [5]:
def save_files(data, output_dir="data"):
    if not data:
        print("Không có dữ liệu để lưu.")
        return

    os.makedirs(output_dir, exist_ok=True)
    fieldnames = data[0].keys()

    # 1. Lưu file CSV
    csv_path = os.path.join(output_dir, "books_data.csv")
    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"Đã lưu: {csv_path}")

In [7]:
if __name__ == "__main__":
    target_url = "http://books.toscrape.com/"
    results = scrape_data(target_url, max_items=10)
    save_files(results)

Đã lưu: data\books_data.csv
